# Survey footprints in the Galactic bulge

In [1]:
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import numpy as np
import pandas as pd
from ipyaladin import Aladin
from astropy.coordinates import Angle, SkyCoord, ICRS
from astropy.io import fits
from astropy.wcs import WCS
from astropy import units as u
from regions import Regions
from selectSIAF import defineApertures, getVertices, computeStcsFootprint, computeRegionFootprint
from getCatalog import gsss_stcsSearchUrl
import pysiaf

import roman_footprint

In [2]:
# Center FOV on Baade's Window
baade = SkyCoord('18:03:32.14', '−30:02:06.96', frame='icrs', unit=(u.hourangle, u.deg))
targetRa = baade.ra
targetDec = baade.dec

In [3]:
# Initialize the aladin viewer - height defines the pixel-size of the viewer window
aladin = Aladin(target=baade, height=600, fov=4.0, survey="P/DSS2/color")

plot_sep_maps = False

## Roman Footprint
Following a procedure laid out in [MAST's notebooks](https://spacetelescope.github.io/mast_notebooks/notebooks/multi_mission/display_footprints/display_footprints.html). 

In [4]:
# Define telescope, instrument, and aperture
selectedTelescope = 'roman'
selectedInstrument = 'WFI'      # Allowed options ALL, WFI, CGI
selectedAperture = 'ALL'        # Allowed options ALL or individual apertures listed in instrument documentation

# Also define the attitude of the telescope during the pointing (angle between 0 and 360)
telescopePositionAngle = 0

In [5]:
# Set up aperture list (=list of detectors) and telescope reference coordinates
apertureList, V2Ref, V3Ref = defineApertures(selectedTelescope, selectedInstrument, selectedAperture)

The field coordinates to display are the survey pointings for the RGBTDS, based on the ROTAC's 2025 recommendations. 

In [6]:
# Set the target field coordinates of the overguide for the RGBTDS, derived fr
rgbtds_fields = [
    SkyCoord(-0.417948, -1.2, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'),
    SkyCoord(-0.008974, -1.2, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'),
    SkyCoord(0.4, -1.2, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'),
    SkyCoord(0.808974, -1.2, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'),
    SkyCoord(1.217948, -1.2, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'),
    SkyCoord(0.0, -0.125, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs')
]

# pysiaf requires separate RA, Dec objects; Aladin takes a string with both RA and Dec, so we separate them here
field_RA = [s.ra for s in rgbtds_fields]
field_Dec = [s.dec for s in rgbtds_fields]

In [7]:
# Compute the telescope attitude matrix
attitude_matrices = []
for i in range(0, len(rgbtds_fields), 1):
    attitude_matrices.append(pysiaf.utils.rotations.attitude_matrix(V2Ref, V3Ref, field_RA[i], field_Dec[i], telescopePositionAngle))

Now we calculate the specific regions of sky in the [IVOA standard STS-S format](https://ivoa.net/documents/STC-S/).  

In [8]:
# Regions are defined as polygon vertices, and the separate regions for individual detectors in an array are 
# accumulated into a single string
combinedSregion = ''
for i in range(0, len(rgbtds_fields), 1):
    attmat = attitude_matrices[i]
    for apertureSiaf in apertureList:
        apertureSiaf.set_attitude_matrix(attmat)
        xVertices, yVertices = getVertices(apertureSiaf)
        
        # Skip PICK which do not have vertices (HST/FGS is only instrument affected)
        if (xVertices is not None and yVertices is not None):
            skyRa, skyDec = apertureSiaf.idl_to_sky(xVertices, yVertices)
            apertureSregion = computeStcsFootprint(apertureSiaf, skyRa, skyDec)
            combinedSregion += apertureSregion

In [9]:
# Initialize the aladin viewer - height defines the pixel-size of the viewer window
if plot_sep_maps:
    aladin = Aladin(target=rgbtds_fields[0], height=600, fov=4.0, survey="P/DSS2/color")

In [10]:
# Overlay the footprints in Roman pink
aladin.add_overlay_from_stcs(combinedSregion, color="#FC74F9") # Pink

## DREAMS Survey

Now overlay pointings from the [DREAMS survey](https://astro.westlake.edu.cn/dreams/introduce/) using Blanco/DECam 

In [11]:
def get_ccd_stcs(ra_ref, dec_ref, dx_min, dy_min):
    # CCD Science dimensions: 2048 x 4096 pixels @ 0.2637"/pixel
    w_deg = (2048 * 0.2637) / 3600.0
    h_deg = (4096 * 0.2637) / 3600.0
    
    # Projection: RA offset depends on Declination
    ra_off = (dx_min / 60.0) / np.cos(np.radians(dec_ref))
    dec_off = dy_min / 60.0
    
    c = [
        (ra_ref - ra_off - w_deg/2, dec_ref + dec_off + h_deg/2),
        (ra_ref - ra_off + w_deg/2, dec_ref + dec_off + h_deg/2),
        (ra_ref - ra_off + w_deg/2, dec_ref + dec_off - h_deg/2),
        (ra_ref - ra_off - w_deg/2, dec_ref + dec_off - h_deg/2)
    ]
    return "POLYGON ICRS " + " ".join([f"{lon:.6f} {lat:.6f}" for lon, lat in c])


In [12]:
# DREAMS survey pointings 
dreams_fields = [
    SkyCoord(268.7, -29.1, frame='icrs', unit=(u.deg, u.deg)),
    SkyCoord(268.1, -29.85, frame='icrs', unit=(u.deg, u.deg))
]

In [13]:
# DECAm detector layout
# Based on data from https://noirlab.edu/science/programs/ctio/instruments/Dark-Energy-Camera/characteristics

# 2. Official Staggered Honeycomb Offsets (in arcminutes)
# Rows are centered around N4/S4 (+/- 4.9). 
# Columns stagger by +/- 9.8 arcmin to create the hexagonal boundary.
y_step = 19.6
x_step = 10.0

# col_idx: [list of row centers in arcmin]
# This mapping follows the 62-CCD science array configuration
decam_layout = {
    0:  [-44.1, -24.5, -4.9, 4.9, 24.5, 44.1],          # Central Column (7 CCDs)
    1:  [-34.3, -14.7, 4.9, 24.5, 44.1],                # Staggered Up (5)
    -1: [-34.3, -14.7, 4.9, 24.5, 44.1],                # Staggered Up (5)
    2:  [-44.1, -24.5, -4.9, 4.9, 24.5, 44.1],          # Back to Central alignment (6)
    -2: [-44.1, -24.5, -4.9, 4.9, 24.5, 44.1],
    3:  [-34.3, -14.7, 4.9, 24.5, 44.1],                # Staggered (5)
    -3: [-34.3, -14.7, 4.9, 24.5, 44.1],
    4:  [-24.5, -4.9, 14.7, 34.3, 53.9],                # Edge Columns (5)
    -4: [-24.5, -4.9, 14.7, 34.3, 53.9],
    5:  [-14.7, 4.9, 24.5, 44.1],                       # Far Edges (4)
    -5: [-14.7, 4.9, 24.5, 44.1],
    6:  [4.9, 24.5],                                    # Outer Tips (2)
    -6: [4.9, 24.5]
}
detector_columns = [-54.9, -44.9, -34.9, -24.9, -14.9, -4.9, 4.9, 14.9, 24.9, 34.9, 44.9, 54.9]
detector_rows = [
    [-18.8, 18.8],     # First row, N30 detector non-functional
    [-28.2, -9.8, 9.8, 28.2], 
    [-37.6, -18.8, 0.0, 18.8, 37.6],
    [-47.0, -28.2, -9.8, 9.8, 28.2, 47.0],    # First of the 3 central double rows 
    [-47.0, -28.2, -9.8, 9.8, 28.2, 47.0],
    [-56.4, -37.6, -18.8, 0.0, 18.8, 37.6, 56.4],  # Central double row
    [-56.4, -37.6, -18.8, 0.0, 18.8, 37.6, 56.4],
    [-47.0, -28.2, -9.8, 9.8, 28.2, 47.0],    # From here the rows are mirrored, but defining explicitly
    [-47.0, -28.2, -9.8, 9.8, 28.2, 47.0],    # allows us to eliminate non-functional detectors
    [-37.6, -18.8, 0.0, 18.8, 37.6],
    [-28.2, -9.8, 9.8, 28.2], 
    [-18.8, 0.0, 18.8],
]

# 3. Render all 62 CCDs
for pointing in dreams_fields:
    for col_idx, col_center in enumerate(detector_columns):
        for row_center in detector_rows[col_idx]:
            stcs_str = get_ccd_stcs(pointing.ra.deg, pointing.dec.deg, col_center, row_center)
            # Style CCD N30 (col 0, row 44.1 approx) differently if desired (since it's not working)
            aladin.add_graphic_overlay_from_stcs(stcs_str, color="#F58E27", lineWidth=1)  # Orange

In [14]:
# Display the aladin viewer 
if plot_sep_maps:
    aladin

## OGLE Fields 
OGLE-IV field pointings and detector layout gathered from [https://ogle.astrouw.edu.pl/](https://ogle.astrouw.edu.pl/)

In [15]:
file_path = '/Users/rstreet/software/rges/rubin_galplane_survey/fields-OGLE-IV-BLG.txt'

ogle_fields  = {}
with open(file_path, 'r') as f:
    file_lines = f.readlines()

    for line in file_lines:
        if line[0:1] != '#':
            entries = line.replace('\n','').split()
            ogle_fields[entries[0]] = SkyCoord(entries[1], entries[2], frame='icrs', unit=(u.hourangle, u.deg))

In [16]:
if plot_sep_maps: 
    aladin = Aladin(target=ogle_fields['BLG501'], height=600, fov=4.0, survey="P/DSS2/color")

In [17]:
def get_ogle_stcs(ra_cen, dec_cen):

    # Coordinate offsets for different corners of the cross-shaped footprint (arcmin -> deg)
    dra_inset = 32.28 / 60.0
    dra_wide = 41.5 / 60.0
    ddec_inset = 18.5 / 60.0
    ddec_wide = 37.0 / 60.0
    
    c = [
        (ra_cen - dra_inset, dec_cen - ddec_wide), # 0
        (ra_cen - dra_inset, dec_cen - ddec_inset), # 1
        (ra_cen - dra_wide, dec_cen - ddec_inset), # 2
        (ra_cen - dra_wide, dec_cen + ddec_inset), # 3
        (ra_cen - dra_inset, dec_cen + ddec_inset), # 4
        (ra_cen - dra_inset, dec_cen + ddec_wide), # 5
        (ra_cen + dra_inset, dec_cen + ddec_wide), # 6
        (ra_cen + dra_inset, dec_cen + ddec_inset), # 7
        (ra_cen + dra_wide, dec_cen + ddec_inset), # 8
        (ra_cen + dra_wide, dec_cen - ddec_inset), # 9
        (ra_cen + dra_inset, dec_cen - ddec_inset), # 10 
        (ra_cen + dra_inset, dec_cen - ddec_wide), # 11
        (ra_cen - dra_inset, dec_cen - ddec_wide)
    ]
    return "POLYGON ICRS " + " ".join([f"{lon:.6f} {lat:.6f}" for lon, lat in c])


In [18]:
high_cadence_fields = [
    500, 501, 504, 505, 506, 534, 507, 508, 534, 535, 645, 646, 511, 513, 507, 535, 600, 661, 609, 603, 
    580, 518, 519, 520, 521, 545, 522, 523, 514, 515, 603, 604, 639, 632, 633, 683, 648, 626, 652, 675, 621, 
    622, 715, 667, 611, 653, 654, 612, 613, 662, 614, 615, 617
]

In [19]:
for field_num in high_cadence_fields:
    field_id = 'BLG' + str(field_num) 
    pointing = ogle_fields[field_id]
    stcs_str = get_ogle_stcs(pointing.ra.deg, pointing.dec.deg)
    print(stcs_str)
    aladin.add_graphic_overlay_from_stcs(stcs_str, color="#FFFFFF", lineWidth=1)  # White

POLYGON ICRS 267.462000 -29.226389 267.462000 -28.918056 267.308333 -28.918056 267.308333 -28.301389 267.462000 -28.301389 267.462000 -27.993056 268.538000 -27.993056 268.538000 -28.301389 268.691667 -28.301389 268.691667 -28.918056 268.538000 -28.918056 268.538000 -29.226389 267.462000 -29.226389
POLYGON ICRS 267.445333 -30.450000 267.445333 -30.141667 267.291667 -30.141667 267.291667 -29.525000 267.445333 -29.525000 267.445333 -29.216667 268.521333 -29.216667 268.521333 -29.525000 268.675000 -29.525000 268.675000 -30.141667 268.521333 -30.141667 268.521333 -30.450000 267.445333 -30.450000
POLYGON ICRS 268.849500 -28.611111 268.849500 -28.302778 268.695833 -28.302778 268.695833 -27.686111 268.849500 -27.686111 268.849500 -27.377778 269.925500 -27.377778 269.925500 -27.686111 270.079167 -27.686111 270.079167 -28.302778 269.925500 -28.302778 269.925500 -28.611111 268.849500 -28.611111
POLYGON ICRS 268.853667 -29.837500 268.853667 -29.529167 268.700000 -29.529167 268.700000 -28.912500 26

In [20]:
if plot_sep_maps: 
    aladin

## MOA/PRIME Survey fields

In [21]:
moa_fields = {
    'GB57': SkyCoord(3.38681, 2.01403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'),
    'GB58': SkyCoord(2.19681, 2.01403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB59': SkyCoord(1.00681, 2.01403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB60': SkyCoord(-0.18319, 2.01403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB61': SkyCoord(-1.37319, 2.01403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB62': SkyCoord(-2.56319, 2.01403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB63': SkyCoord(-3.75319, 2.01403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB74': SkyCoord(3.38681, 0.82403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB75': SkyCoord(2.19681, 0.82403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB76': SkyCoord(1.00681, 0.82403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB77': SkyCoord(-0.18319, 0.82403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB78': SkyCoord(-1.37319, 0.82403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB79': SkyCoord(-2.56319, 0.82403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB80': SkyCoord(-3.75319, 0.82403, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB91': SkyCoord(3.38681, -0.36597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB92': SkyCoord(2.19681, -0.36597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB93': SkyCoord(1.00681, -0.36597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB94': SkyCoord(-0.18319, -0.36597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB95': SkyCoord(-1.37319, -0.36597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB96': SkyCoord(-2.56319, -0.36597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB97': SkyCoord(-3.75319, -0.36597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB108': SkyCoord(3.38681, -1.55597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB109': SkyCoord(2.19681, -1.55597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'),
    'GB110': SkyCoord(1.00681, -1.55597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB111': SkyCoord(-0.18319, -1.55597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB112': SkyCoord(-1.37319, -1.55597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB113': SkyCoord(-2.56319, -1.55597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB114': SkyCoord(-3.75319, -1.55597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB125': SkyCoord(3.38681, -2.74597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB126': SkyCoord(2.19681, -2.74597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB127': SkyCoord(1.00681, -2.74597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB128': SkyCoord(-0.18319, -2.74597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB129': SkyCoord(-1.37319, -2.74597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB130': SkyCoord(-2.56319, -2.74597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB131': SkyCoord(-3.75319, -2.74597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'),
    'GB124': SkyCoord(4.57681, -2.74597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB141': SkyCoord(4.57681, -3.93597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB142': SkyCoord(3.38681, -3.93597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB143': SkyCoord(2.19681, -3.93597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB144': SkyCoord(1.00681, -3.93597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB145': SkyCoord(-0.18319, -3.93597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB146': SkyCoord(-1.37319, -3.93597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB147': SkyCoord(-2.56319, -3.93597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs'), 
    'GB148': SkyCoord(-3.75319, -3.93597, frame='galactic', unit=(u.deg, u.deg)).transform_to('icrs')
}

In [22]:
def get_square_stcs(ra_cen, dec_cen, width):
    
    dra = width / 2.0 # degrees
    ddec = width / 2.0 # degrees
    
    c = [
        (ra_cen - dra, dec_cen - ddec), # 0
        (ra_cen - dra, dec_cen + ddec), # 1
        (ra_cen + dra, dec_cen + ddec), # 2
        (ra_cen + dra, dec_cen - ddec), # 3
        (ra_cen - dra, dec_cen - ddec)
    ]
    return "POLYGON ICRS " + " ".join([f"{lon:.6f} {lat:.6f}" for lon, lat in c])


In [23]:
if plot_sep_maps: 
    aladin = Aladin(target=moa_fields['GB57'], height=600, fov=4.0, survey="P/DSS2/color")

In [24]:
# MOA/PRIME's field of view is a 2x2 square mosaic which we represent as a square to avoid too many lines on the plot
for field_id, pointing in moa_fields.items():
    stcs_str = get_square_stcs(pointing.ra.deg, pointing.dec.deg, 1.2)
    print(stcs_str)
    aladin.add_graphic_overlay_from_stcs(stcs_str, color="#FFFF73", lineWidth=1) # Yellow

POLYGON ICRS 265.853329 -25.596607 265.853329 -24.396607 267.053329 -24.396607 267.053329 -25.596607 265.853329 -25.596607
POLYGON ICRS 265.165078 -26.610755 265.165078 -25.410755 266.365078 -25.410755 266.365078 -26.610755 265.165078 -26.610755
POLYGON ICRS 264.464027 -27.621129 264.464027 -26.421129 265.664027 -26.421129 265.664027 -27.621129 264.464027 -27.621129
POLYGON ICRS 263.749442 -28.627503 263.749442 -27.427503 264.949442 -27.427503 264.949442 -28.627503 263.749442 -28.627503
POLYGON ICRS 263.020558 -29.629642 263.020558 -28.429642 264.220558 -28.429642 264.220558 -29.629642 263.020558 -29.629642
POLYGON ICRS 262.276573 -30.627295 262.276573 -29.427295 263.476573 -29.427295 263.476573 -30.627295 262.276573 -30.627295
POLYGON ICRS 261.516648 -31.620198 261.516648 -30.420198 262.716648 -30.420198 262.716648 -31.620198 261.516648 -31.620198
POLYGON ICRS 266.980675 -26.210928 266.980675 -25.010928 268.180675 -25.010928 268.180675 -26.210928 266.980675 -26.210928
POLYGON ICRS 266

In [25]:
if plot_sep_maps: 
    aladin

## KMTNet Survey 

Field maps were derived from [this paper](https://arxiv.org/pdf/1703.06883)

In [26]:
kmtnet_fields = {
    1: SkyCoord(268.5, -31.25, frame='icrs', unit=(u.deg, u.deg)),  # Southern Bulge fields
    2: SkyCoord(268.5, -29.0, frame='icrs', unit=(u.deg, u.deg)),
    3: SkyCoord(270.5, -28.0, frame='icrs', unit=(u.deg, u.deg)),
    4: SkyCoord(271.0, -30.0, frame='icrs', unit=(u.deg, u.deg)),
    35: SkyCoord(271.1, -26.0, frame='icrs', unit=(u.deg, u.deg)),
    31: SkyCoord(273.25, -26.0, frame='icrs', unit=(u.deg, u.deg)),
    36: SkyCoord(274.0, -23.8, frame='icrs', unit=(u.deg, u.deg)),
    32: SkyCoord(273.0, -28.0, frame='icrs', unit=(u.deg, u.deg)),
    33: SkyCoord(273.25, -30.0, frame='icrs', unit=(u.deg, u.deg)),
    34: SkyCoord(271.0, -32.25, frame='icrs', unit=(u.deg, u.deg)),
    22: SkyCoord(268.5, -33.3, frame='icrs', unit=(u.deg, u.deg)),
    21: SkyCoord(268.5, -35.4, frame='icrs', unit=(u.deg, u.deg)),
    37: SkyCoord(266.2, -35.4, frame='icrs', unit=(u.deg, u.deg)),
    17: SkyCoord(266.2, -33.25, frame='icrs', unit=(u.deg, u.deg)),
    13: SkyCoord(263.75, -34.4, frame='icrs', unit=(u.deg, u.deg)),
    38: SkyCoord(269.75, -22.5, frame='icrs', unit=(u.deg, u.deg)),  # Northern Bulge
    20: SkyCoord(266.5, -22.5, frame='icrs', unit=(u.deg, u.deg)),
    19: SkyCoord(266.5, -24.5, frame='icrs', unit=(u.deg, u.deg)),
    16: SkyCoord(264.2, -24.5, frame='icrs', unit=(u.deg, u.deg)),
    15: SkyCoord(264.2, -26.75, frame='icrs', unit=(u.deg, u.deg)),
    18: SkyCoord(266.5, -26.75, frame='icrs', unit=(u.deg, u.deg)),
    12: SkyCoord(262.0, -27.8, frame='icrs', unit=(u.deg, u.deg)),
    14: SkyCoord(264.2, -28.9, frame='icrs', unit=(u.deg, u.deg)),
    11: SkyCoord(262.0, -30.0, frame='icrs', unit=(u.deg, u.deg)),
    41: SkyCoord(268.5, -31.5, frame='icrs', unit=(u.deg, u.deg)),
    42: SkyCoord(268.5, -29.25, frame='icrs', unit=(u.deg, u.deg)),
    43: SkyCoord(270.5, -28.25, frame='icrs', unit=(u.deg, u.deg)),
}

In [27]:
if plot_sep_maps: 
    aladin = Aladin(target=kmtnet_fields[1], height=600, fov=4.0, survey="P/DSS2/color")

In [28]:
# Represent KMTNet fields as 2x2deg squares.
for field_id, pointing in kmtnet_fields.items():
    stcs_str = get_square_stcs(pointing.ra.deg, pointing.dec.deg, 2.0)
    print(stcs_str)
    aladin.add_graphic_overlay_from_stcs(stcs_str, color="#73FFF1", lineWidth=1)  # Cyan

POLYGON ICRS 267.500000 -32.250000 267.500000 -30.250000 269.500000 -30.250000 269.500000 -32.250000 267.500000 -32.250000
POLYGON ICRS 267.500000 -30.000000 267.500000 -28.000000 269.500000 -28.000000 269.500000 -30.000000 267.500000 -30.000000
POLYGON ICRS 269.500000 -29.000000 269.500000 -27.000000 271.500000 -27.000000 271.500000 -29.000000 269.500000 -29.000000
POLYGON ICRS 270.000000 -31.000000 270.000000 -29.000000 272.000000 -29.000000 272.000000 -31.000000 270.000000 -31.000000
POLYGON ICRS 270.100000 -27.000000 270.100000 -25.000000 272.100000 -25.000000 272.100000 -27.000000 270.100000 -27.000000
POLYGON ICRS 272.250000 -27.000000 272.250000 -25.000000 274.250000 -25.000000 274.250000 -27.000000 272.250000 -27.000000
POLYGON ICRS 273.000000 -24.800000 273.000000 -22.800000 275.000000 -22.800000 275.000000 -24.800000 273.000000 -24.800000
POLYGON ICRS 272.000000 -29.000000 272.000000 -27.000000 274.000000 -27.000000 274.000000 -29.000000 272.000000 -29.000000
POLYGON ICRS 272

In [29]:
aladin